# Обучение YOLO для оценки заполненности мусорных контейнеров

Перед запуском включите GPU: **Runtime → Change runtime type → T4 GPU**. Используйте архив `trash_bin_dataset.zip`, подготовленный локальным скриптом `scripts/prepare_dataset.py` после успешной проверки.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU не обнаружен. Включите T4 GPU и перезапустите среду.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT = '/content/drive/MyDrive/trash-bin-ai'
# Вариант A: положите сюда архив, подготовленный локально.
# Вариант B: скопируйте папку data/merged в Google Drive/trash-bin-ai/dataset_merged.
ARCHIVE_PATH = f'{DRIVE_PROJECT}/trash_bin_dataset.zip'
FOLDER_PATH = f'{DRIVE_PROJECT}/dataset_merged'
EXPECTED_IMAGES = {'train': 627, 'val': 134, 'test': 135}
EXPECTED_BOXES = {'empty': 768, 'partial': 117, 'full': 634, 'overflow': 69}

In [ ]:
# Необязательно: загрузка архива непосредственно с компьютера.
# Иначе будет использован ARCHIVE_PATH на Google Drive.
from google.colab import files
from pathlib import Path
UPLOAD_FROM_COMPUTER = False
SOURCE_ARCHIVE = None
if UPLOAD_FROM_COMPUTER:
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Загрузите ровно один архив.'
    uploaded_name = next(iter(uploaded))
    assert uploaded_name == 'trash_bin_dataset.zip', f'Ожидается trash_bin_dataset.zip, получен {uploaded_name}'
    SOURCE_ARCHIVE = Path('/content') / uploaded_name
else:
    SOURCE_ARCHIVE = Path(ARCHIVE_PATH) if Path(ARCHIVE_PATH).is_file() else None

In [ ]:
!pip -q install -U ultralytics pyyaml onnx
import os, shutil, zipfile
from pathlib import Path
import yaml

WORK_DIR = Path('/content/trash-bin-ai')
EXTRACT_DIR = WORK_DIR / 'dataset'
shutil.rmtree(WORK_DIR, ignore_errors=True)
EXTRACT_DIR.mkdir(parents=True)
if SOURCE_ARCHIVE is not None:
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        archive.extractall(EXTRACT_DIR)
elif Path(FOLDER_PATH).is_dir():
    shutil.copytree(FOLDER_PATH, EXTRACT_DIR / 'merged')
else:
    raise FileNotFoundError('Не найден архив или папка на Google Drive. См. ячейку выше.')

candidates = [path.parent for path in EXTRACT_DIR.rglob('data.yaml') if (path.parent / 'images').is_dir() and (path.parent / 'labels').is_dir()]
assert len(candidates) == 1, f'Ожидалась одна папка датасета, найдено: {candidates}'
DATASET_DIR = candidates[0]
print('Dataset:', DATASET_DIR)

In [ ]:
# Строгая проверка структуры и утверждённой статистики до обучения.
expected_names = {0: 'empty', 1: 'partial', 2: 'full', 3: 'overflow'}
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
actual_boxes = {name: 0 for name in expected_names.values()}
for split in ('train', 'val', 'test'):
    images = {path.stem: path for path in (DATASET_DIR / 'images' / split).iterdir() if path.suffix.casefold() in image_extensions}
    labels = {path.stem: path for path in (DATASET_DIR / 'labels' / split).glob('*.txt')}
    assert images, f'Пустой images/{split}'
    assert images.keys() == labels.keys(), f'Непарные images/labels в {split}'
    assert len(images) == EXPECTED_IMAGES[split], f'Неверное число изображений в {split}: {len(images)}'
    for label in labels.values():
        lines = label.read_text(encoding='utf-8').splitlines()
        assert lines, f'Пустой label: {label}'
        for line_number, line in enumerate(lines, start=1):
            fields = line.split()
            assert len(fields) == 5, f'Некорректный label: {label}:{line_number}'
            class_id = int(fields[0])
            x, y, width, height = map(float, fields[1:])
            assert class_id in expected_names, f'Неизвестный класс: {label}:{line_number}'
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1 and x - width / 2 >= 0 and x + width / 2 <= 1 and y - height / 2 >= 0 and y + height / 2 <= 1, f'BBox вне изображения: {label}:{line_number}'
            actual_boxes[expected_names[class_id]] += 1
    print(split, 'images=', len(images), 'labels=', len(labels))
assert actual_boxes == EXPECTED_BOXES, f'Неверная статистика рамок: {actual_boxes}'
print('Подтверждённые рамки:', actual_boxes)
source_yaml = yaml.safe_load((DATASET_DIR / 'data.yaml').read_text())
assert source_yaml.get('names') == expected_names or source_yaml.get('names') == list(expected_names.values())
colab_yaml = {
    'path': str(DATASET_DIR),
    'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'nc': 4, 'names': expected_names,
}
COLAB_YAML = WORK_DIR / 'data_colab.yaml'
COLAB_YAML.write_text(yaml.safe_dump(colab_yaml, sort_keys=False))
print(COLAB_YAML.read_text())

In [ ]:
from ultralytics import YOLO

# YOLO26 — предпочтительная компактная модель. Если установленная версия
# Ultralytics не умеет её загрузить, автоматически используется YOLO11.
try:
    model = YOLO('yolo26n.pt')
    BASE_MODEL = 'yolo26n.pt'
except Exception as exc:
    print(f'yolo26n.pt недоступна/несовместима ({type(exc).__name__}); используется yolo11n.pt')
    model = YOLO('yolo11n.pt')
    BASE_MODEL = 'yolo11n.pt'
print('Transfer-learning base:', BASE_MODEL)

In [ ]:
results = model.train(
    data=str(COLAB_YAML),
    epochs=60,
    patience=12,
    imgsz=640,
    batch=-1,
    device=0,
    workers=2,
    pretrained=True,
    project=str(WORK_DIR / 'runs'),
    name='trash_bin_fill_level',
    exist_ok=True,
)

In [ ]:
from IPython.display import Image as DisplayImage, display

RUN_DIR = WORK_DIR / 'runs' / 'trash_bin_fill_level'
BEST = RUN_DIR / 'weights' / 'best.pt'
assert BEST.is_file(), f'Не найден {BEST}'
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(COLAB_YAML), split='test', imgsz=640, plots=True)
for filename in ('results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png'):
    path = RUN_DIR / filename
    if path.exists():
        display(DisplayImage(filename=str(path)))
print(metrics)

In [ ]:
predictions = best_model.predict(
    source=str(DATASET_DIR / 'images' / 'test'),
    imgsz=640, conf=0.25, save=True, project=str(WORK_DIR / 'predictions'), name='test', exist_ok=True
)
prediction_dir = WORK_DIR / 'predictions' / 'test'
for path in list(prediction_dir.glob('*'))[:8]:
    display(DisplayImage(filename=str(path)))

In [ ]:
# Сохраняем веса и экспортируем варианты для Raspberry Pi.
EXPORT_DIR = Path(DRIVE_PROJECT) / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LAST = RUN_DIR / 'weights' / 'last.pt'
assert LAST.is_file(), f'Не найден {LAST}'
shutil.copy2(BEST, EXPORT_DIR / 'best.pt')
shutil.copy2(LAST, EXPORT_DIR / 'last.pt')
onnx_path = Path(best_model.export(format='onnx', imgsz=640))
ncnn_path = Path(best_model.export(format='ncnn', imgsz=640))
for artifact in (onnx_path, ncnn_path):
    destination = EXPORT_DIR / artifact.name
    if artifact.is_dir():
        shutil.copytree(artifact, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(artifact, destination)
print('best.pt:', EXPORT_DIR / 'best.pt')
print('last.pt:', EXPORT_DIR / 'last.pt')
print('ONNX:', onnx_path)
print('NCNN:', ncnn_path)
print('Все экспорты сохранены в:', EXPORT_DIR)

In [ ]:
# Необязательно: проверка на собственной фотографии.
# from google.colab import files
# uploaded = files.upload()
# photo = next(iter(uploaded))
# best_model.predict(source=photo, imgsz=640, conf=0.25, show=True)